# Chess AI — Self-Play Only (Kaggle)

**Dùng khi:** Đã có model (từ PGN training hoặc self-play trước đó),  
muốn tiếp tục cải thiện bằng self-play mà **không** cần train PGN lại.

**Workflow:**
1. **Cell 1** — Cài đặt & Cấu hình
2. **Cell 2** — Bootstrap: tìm source code + model hiện tại
3. **Cell 3** — Self-Play Pipeline (vòng lặp vô hạn)

### Datasets cần add:
| Dataset | Chứa gì | Mount path |
|---------|---------|------------|
| `chess-ai-source` | Source code (engine C++, pipeline.py) | `/kaggle/input/chess-ai-source/` |
| `chess-model` | `best_model_traced.pt` từ lần train trước | `/kaggle/input/chess-model/` |

> **Lưu ý:** Nếu không add `chess-model`, pipeline sẽ tự tạo model ngẫu nhiên.


In [ ]:
# ─── Cell 1: Cài đặt & Cấu hình Self-Play ────────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "chess", "-q"])

from pathlib import Path

# ┌─────────────────────────────────────────────────────────────────────────┐
# │  CHỈNH Ở ĐÂY                                                            │
# └─────────────────────────────────────────────────────────────────────────┘
SOURCE_DATASET = "/kaggle/input/chess-ai-source"   # Dataset chứa source code
MODEL_DATASET  = "/kaggle/input/chess-model"       # Dataset chứa best_model_traced.pt
                                                    # (để trống nếu không có)

# ── Self-Play Hyperparameters ────────────────────────────────────────────────
SP_SIMULATIONS    = 400   # MCTS simulations/nước  (tăng = mạnh hơn nhưng chậm hơn)
SP_GAMES_PER_GEN  = 100   # Số game mỗi generation
SP_EPOCHS         = 3     # Epoch train sau mỗi gen
SP_BATCH_SIZE     = 256   # Batch size khi train
SP_LR             = 1e-3  # Learning rate
SP_MAX_GEN        = None  # None = chạy vô hạn cho đến hết thời gian Kaggle
SP_TEMPERATURE_MOVES = 50 # Số nước đầu dùng temperature sampling (exploration)
SP_RESIGN_THRESH  = 0.9   # Resign nếu value < -0.9 (giảm game vô ích)
SP_MIN_RESIGN_PLY = 20    # Không resign trước ply này

# ── Paths ────────────────────────────────────────────────────────────────────
WORKDIR    = Path("/kaggle/working/chess_selfplay")
OUTPUT_DIR = Path("/kaggle/working/chess_outputs")   # Replay + models lưu ở đây
WORKDIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "checkpoints").mkdir(exist_ok=True)

print("✓ Cell 1 done — Cấu hình:")
print(f"  simulations={SP_SIMULATIONS}  games/gen={SP_GAMES_PER_GEN}  epochs/gen={SP_EPOCHS}")
print(f"  max_gen={'∞' if SP_MAX_GEN is None else SP_MAX_GEN}  temperature_moves={SP_TEMPERATURE_MOVES}")
print(f"  resign_thresh={SP_RESIGN_THRESH}  min_resign_ply={SP_MIN_RESIGN_PLY}")
print(f"  OUTPUT: {OUTPUT_DIR}")


In [ ]:
# ─── Cell 2: Bootstrap — tìm source code & model ──────────────────────────────
import glob, shutil
from pathlib import Path

# ── Tìm PROJECT_ROOT ─────────────────────────────────────────────────────────
def find_project_root(dataset_path: str) -> Path:
    for pattern in [
        f"{dataset_path}/Chess-AI-out",
        f"{dataset_path}/*/Chess-AI-out",
        f"{dataset_path}",
    ]:
        for m in glob.glob(pattern):
            p = Path(m)
            if (p / "AI/engine/selfplay.cpp").exists() or \
               (p / "AI/src/colab_selfplay_pipeline.py").exists():
                return p
    raise FileNotFoundError(f"Không tìm thấy project root trong {dataset_path}")

try:
    PROJECT_ROOT = find_project_root(SOURCE_DATASET)
    print(f"[Bootstrap] ✓ PROJECT_ROOT = {PROJECT_ROOT}")
except FileNotFoundError as e:
    print(f"[ERROR] {e}")
    print("[HELP] Hãy thêm dataset chứa source code Chess AI vào kaggle")
    raise

# ── Add pipeline vào sys.path ─────────────────────────────────────────────────
import sys
pipeline_dir = PROJECT_ROOT / "AI" / "src"
if str(pipeline_dir) not in sys.path:
    sys.path.insert(0, str(pipeline_dir))
print(f"[Bootstrap] Pipeline dir: {pipeline_dir}")

# ── Tìm model khởi đầu ───────────────────────────────────────────────────────
INITIAL_MODEL = None
model_candidates = []

# 1. Từ model dataset
if Path(MODEL_DATASET).exists():
    model_candidates += [
        f"{MODEL_DATASET}/best_model_traced.pt",
        f"{MODEL_DATASET}/chess_pgn_train/best_model_traced.pt",
    ]
    model_candidates += sorted(glob.glob(f"{MODEL_DATASET}/**/*.pt", recursive=True))

# 2. Từ source dataset (nếu đã save model vào đó)
model_candidates += [
    f"{SOURCE_DATASET}/Chess-AI-out/AI/data/best_model_traced.pt",
    f"{SOURCE_DATASET}/best_model_traced.pt",
]

# 3. Từ output lần trước
model_candidates += sorted(
    glob.glob(str(OUTPUT_DIR / "model_gen_*.pt")),
    key=lambda p: int(Path(p).stem.split("_")[-1]) if Path(p).stem.split("_")[-1].isdigit() else 0,
    reverse=True
)

for c in model_candidates:
    if Path(c).exists():
        INITIAL_MODEL = Path(c)
        size_mb = INITIAL_MODEL.stat().st_size / 1e6
        print(f"[Bootstrap] ✓ Model tìm thấy: {INITIAL_MODEL}  ({size_mb:.1f}MB)")
        break

if INITIAL_MODEL is None:
    print("[Bootstrap] ⚠️  Không tìm thấy model → self-play sẽ dùng random init")
    print("[Bootstrap] Lưu ý: Model random sẽ chơi rất tệ ở đầu, cần nhiều gen để cải thiện")

# Kiểm tra resume state
resume_state_path = OUTPUT_DIR / "resume_state.json"
if resume_state_path.exists():
    import json
    with open(resume_state_path) as f:
        rs = json.load(f)
    print(f"\n[Bootstrap] ✓ Resume state tìm thấy: gen={rs.get('generation', 0)}")
    print(f"[Bootstrap]   Best model: {rs.get('best_model_path', 'N/A')}")
    RESUME = True
else:
    print("\n[Bootstrap] Chạy mới (không có resume state)")
    RESUME = False

print("\n✓ Cell 2 done")


In [ ]:
# ─── Cell 3: Self-Play Pipeline ───────────────────────────────────────────────
# Pipeline vòng lặp:
#   1. Build C++ engine (tự động, cache nếu source không đổi)
#   2. Self-play N games → replay buffer .bin
#   3. Train policy+value net trên replay buffer
#   4. Export TorchScript model mới
#   5. Lặp lại từ bước 2 với model mới
#
# Dừng tự nhiên khi Kaggle timeout (~9h) hoặc đủ SP_MAX_GEN generations.
# Lần sau: thêm OUTPUT_DIR vào dataset → Resume tự động từ gen đã làm.
#
# Output files:
#   OUTPUT_DIR/selfplay_gen_N.bin     — replay data gen N
#   OUTPUT_DIR/model_gen_N.pt         — model sau gen N
#   OUTPUT_DIR/best_model_traced.pt   — model tốt nhất hiện tại
#   OUTPUT_DIR/checkpoint_gen_N.pt    — checkpoint để resume
#   OUTPUT_DIR/resume_state.json      — resume state

import importlib.util
from pathlib import Path

spec = importlib.util.spec_from_file_location(
    "colab_selfplay_pipeline",
    PROJECT_ROOT / "AI" / "src" / "colab_selfplay_pipeline.py"
)
pipeline = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pipeline)

print(f"\n{'='*60}")
print(f"SELF-PLAY PIPELINE — {'Resume' if RESUME else 'Fresh start'}")
print(f"  simulations   = {SP_SIMULATIONS}")
print(f"  games/gen     = {SP_GAMES_PER_GEN}")
print(f"  epochs/gen    = {SP_EPOCHS}")
print(f"  max_gen       = {'∞' if SP_MAX_GEN is None else SP_MAX_GEN}")
print(f"  temp_moves    = {SP_TEMPERATURE_MOVES}")
print(f"  output        = {OUTPUT_DIR}")
print(f"{'='*60}\n")

pipeline.run_pipeline(
    project_root             = PROJECT_ROOT,
    workdir                  = WORKDIR,
    drive_root               = OUTPUT_DIR,
    best_model_path_override = WORKDIR / "best_model_traced.pt",
    initial_model_path       = INITIAL_MODEL,
    simulations              = SP_SIMULATIONS,
    games_per_generation     = SP_GAMES_PER_GEN,
    epochs                   = SP_EPOCHS,
    batch_size               = SP_BATCH_SIZE,
    learning_rate            = SP_LR,
    max_generations          = SP_MAX_GEN,
    infinite                 = (SP_MAX_GEN is None),
    resume                   = RESUME,
    temperature_moves        = SP_TEMPERATURE_MOVES,
    log_every_games          = max(5, SP_GAMES_PER_GEN // 20),
    heartbeat_seconds        = 60,
    resign_thresh            = SP_RESIGN_THRESH,
    min_resign_ply           = SP_MIN_RESIGN_PLY,
)
